# 🧛 Vampire Restyle — Colab Training (Proof-of-Concept)

Trains the **Stage-1 vampire model** (SDXL-Turbo + LoRA + InstantID) for the attention-trigger
site, distilled from the live Fal `ip-adapter-face-id` teacher.

> ## ⚠️ LICENSE GUARDRAIL — READ THIS
> This notebook uses **FFHQ**, which is **NON-COMMERCIAL / research only**. The weights it
> produces are a **PROOF-OF-CONCEPT**: they validate the pipeline + quality, but they **MUST
> NOT ship on the live Cobblestone client site**. Before production, re-run the full pipeline
> with `--from-dir` pointed at a **cleared/consented** face set. Do not skip this.

## 1. Check GPU
Runtime → Change runtime type → **GPU** (T4 is fine for the smoke test; A100/L4 for the full run).

In [ ]:
!nvidia-smi

## 2. Get the repo + install deps
Push `~/Documents/vampire model` to a git repo and set `REPO_URL`, **or** upload it as a zip
(uncomment the upload cell). Then install requirements.

In [ ]:
REPO_URL = ""  # e.g. "https://github.com/<you>/vampire-model.git"

import os
if REPO_URL:
    !git clone $REPO_URL vampire_model
    %cd vampire_model
else:
    # --- OR upload a zip of the repo instead of cloning ---
    from google.colab import files
    up = files.upload()              # choose your vampire-model.zip
    import zipfile
    z = list(up)[0]
    zipfile.ZipFile(z).extractall("vampire_model")
    %cd vampire_model

!pip -q install -r requirements.txt

## 3. Secrets — Fal key
Used to call the teacher (`ip-adapter-face-id`) to generate (face → vampire) target pairs.
Incurs Fal cost. Pasted via getpass so it isn't saved in the notebook.

In [ ]:
import os, getpass
os.environ["FAL_KEY"] = getpass.getpass("FAL_KEY: ")
os.environ["FAL_API_KEY"] = os.environ["FAL_KEY"]

## 4. Fetch FFHQ (POC dataset)
Small subset for the smoke test. **Verify the HF repo ID exists** (they move) — swap if it 404s.
Use a 256px+ mirror for the full run; 64px is fine only for the smoke test.

In [ ]:
# verify this repo on huggingface.co; swap if needed
HF_REPO = "nuwandaa/ffhq256"
!python src/fetch_ffhq.py --hf-repo "$HF_REPO" --n 50 --out data/faces --res 512
!ls data/faces | head

## 5. 🔬 SMOKE TEST FIRST  (do not skip)
Runs the **entire** pipeline on ~10 images in a few minutes for a couple dollars of Fal spend.
Proves the plumbing before you commit GPU/Fal budget to the full run. Must end in ✅.

In [ ]:
!python src/smoke_test.py --config config.yaml --faces data/faces --n 10

## 6. ✅ Full run — only if the smoke test passed
### 6a. Generate the full teacher dataset
Tune `--limit` (start 500–2000). This is the main Fal spend; `--sleep` rate-limits if needed.

In [ ]:
!python src/teacher_generate.py --faces data/faces --out data/targets --pairs data/pairs.jsonl --config config.yaml --limit 2000

### 6b. Train the Stage-1 vampire LoRA

In [ ]:
!python src/train_lora.py --config config.yaml

### 6c. Validate against acceptance bars
ArcFace cosine ≥ 0.65, landmark RMSE ≤ 6px, latency. Fill `EVAL.md` from this.

In [ ]:
!python src/eval.py --config config.yaml --model stage1 --faces data/faces

## 7. Download the artifacts (LoRA + eval)

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("vampire_stage1_artifacts", "zip", "checkpoints")
files.download("vampire_stage1_artifacts.zip")

## 8. 🛑 Before you ship — production guardrail
These weights were trained on **FFHQ (non-commercial)** = **proof-of-concept only**.

To produce shippable weights:
1. Assemble a **cleared/consented** face set.
2. Re-run from step 4 with `--from-dir <your_cleared_faces>` instead of `--hf-repo`.
3. Re-validate (step 6c), then hand the new weights to **openclaw** per
   `INTEGRATION-FOR-OPENCLAW.md` (Track B: stand up `infer_server.py`, point `STAGE1_URL`,
   flip the engine to PRIMARY with `ip-adapter-face-id` as fallback).

Identity is the headline feature — don't ship anything that fails the ArcFace bar.